In [2]:
# ============================================================================
# CHURN PREDICTION - CLEAN PRODUCTION CODE
# ============================================================================

import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, confusion_matrix, precision_recall_curve
)
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# ============================================================================
# LOAD DATA
# ============================================================================

path = kagglehub.dataset_download("blastchar/telco-customer-churn")
csv_path = os.path.join(path, [f for f in os.listdir(path) if f.endswith('.csv')][0])
df = pd.read_csv(csv_path)

print(f"Data loaded: {df.shape[0]} customers, {df.shape[1]} features")

# ============================================================================
# EDA
# ============================================================================

churn_rate = (df['Churn'] == 'Yes').sum() / len(df) * 100

print(f"Churn rate: {churn_rate:.1f}%")
print(f"\nChurn by contract type:")
for contract, rate in df.groupby('Contract')['Churn'].apply(
    lambda x: (x == 'Yes').sum() / len(x) * 100).items():
    print(f"  {contract}: {rate:.1f}%")

# ============================================================================
# DATA PREP
# ============================================================================

df_processed = df.copy()
df_processed['Churn'] = (df_processed['Churn'] == 'Yes').astype(int)
df_processed['TotalCharges'] = pd.to_numeric(df_processed['TotalCharges'], 
                                              errors='coerce').fillna(0)

categorical_cols = [col for col in df.select_dtypes(include=['object']).columns 
                    if col not in ['customerID', 'Churn']]

df_processed = pd.get_dummies(df_processed, columns=categorical_cols, drop_first=True)
df_processed = df_processed.drop('customerID', axis=1)

df_processed['is_new_customer'] = (df['tenure'] < 6).astype(int)
df_processed['tenure_years'] = df['tenure'] / 12
df_processed = df_processed.drop('tenure', axis=1)

X = df_processed.drop('Churn', axis=1).reset_index(drop=True)
y = df_processed['Churn'].reset_index(drop=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns)

# ============================================================================
# BASELINE MODEL
# ============================================================================

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)
y_test_pred_lr = lr.predict(X_test_scaled)

print(f"\nLogistic Regression - Recall: {recall_score(y_test, y_test_pred_lr):.1%}")

# ============================================================================
# RANDOM FOREST
# ============================================================================

print("Training RandomForest...")
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42,
                            class_weight='balanced', n_jobs=-1)
rf.fit(X_train, y_train)

y_test_pred_rf = rf.predict(X_test)
y_test_pred_proba_rf = rf.predict_proba(X_test)[:, 1]

print(f"RandomForest - Recall: {recall_score(y_test, y_test_pred_rf):.1%}, ROC-AUC: {roc_auc_score(y_test, y_test_pred_proba_rf):.3f}")

# Feature importance
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\nTop 3 Features:")
for idx, (_, row) in enumerate(feature_importance.head(3).iterrows(), 1):
    print(f"  {idx}. {row['feature']}: {row['importance']:.4f}")

# PLOT 1
plt.figure(figsize=(10, 6))
plt.barh(feature_importance.head(10)['feature'], feature_importance.head(10)['importance'], color='steelblue')
plt.xlabel('Importance')
plt.title('Top 10 Features Driving Churn')
plt.tight_layout()
plt.savefig('01_feature_importance.png', dpi=100, bbox_inches='tight')
plt.close()

# ============================================================================
# THRESHOLD OPTIMIZATION
# ============================================================================

COST_OF_OFFER = 500
VALUE_IF_RETAINED = 2000
VALUE_IF_LOST = 2000

thresholds_list = np.arange(0.1, 1.0, 0.05)
business_values = []
target_counts = []
precisions = []
recalls = []

for threshold in thresholds_list:
    y_pred_threshold = (y_test_pred_proba_rf > threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_threshold).ravel()
    
    value = tp * (VALUE_IF_RETAINED - COST_OF_OFFER) - fp * COST_OF_OFFER - fn * VALUE_IF_LOST
    
    business_values.append(value)
    target_counts.append(y_pred_threshold.sum())
    precisions.append(precision_score(y_test, y_pred_threshold, zero_division=0))
    recalls.append(recall_score(y_test, y_pred_threshold, zero_division=0))

optimal_idx = np.argmax(business_values)
optimal_threshold = thresholds_list[optimal_idx]
optimal_value = business_values[optimal_idx]

print(f"\nOptimal Threshold: {optimal_threshold:.2f}")
print(f"Customers to target: {target_counts[optimal_idx]}")
print(f"Precision: {precisions[optimal_idx]:.1%}, Recall: {recalls[optimal_idx]:.1%}")
print(f"Net business value: ₹{optimal_value:,.0f}")

# PLOT 2
plt.figure(figsize=(10, 6))
precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_test_pred_proba_rf)
plt.plot(recall_vals, precision_vals, linewidth=2)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Tradeoff')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('02_precision_recall.png', dpi=100, bbox_inches='tight')
plt.close()

# PLOT 3
plt.figure(figsize=(10, 6))
plt.plot(thresholds_list, business_values, marker='o', linewidth=2, markersize=8, color='green')
plt.axvline(optimal_threshold, color='red', linestyle='--', linewidth=2, label=f'Optimal ({optimal_threshold:.2f})')
plt.axvline(0.5, color='gray', linestyle='--', linewidth=1, label='Default (0.5)')
plt.xlabel('Decision Threshold')
plt.ylabel('Net Business Value (₹)')
plt.title('Business Value by Threshold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('03_business_value.png', dpi=100, bbox_inches='tight')
plt.close()

# ============================================================================
# EXPLAIN TOP PREDICTIONS
# ============================================================================

print(f"\nTop 5 Customers at Risk:")

churn_probs = rf.predict_proba(X_test)[:, 1]
top_5_indices = np.argsort(churn_probs)[-5:][::-1]

for rank, idx in enumerate(top_5_indices, 1):
    prob = churn_probs[idx]
    actual = y_test.iloc[idx]
    print(f"  {rank}. Probability: {prob:.1%} | Actual: {'Churned' if actual else 'Retained'}")

# PLOT 4
plt.figure(figsize=(10, 6))
plt.barh(feature_importance.head(12)['feature'], feature_importance.head(12)['importance'], color='steelblue')
plt.xlabel('Feature Importance Score')
plt.title('Feature Importance for Churn Prediction')
plt.tight_layout()
plt.savefig('04_feature_contribution.png', dpi=100, bbox_inches='tight')
plt.close()

# ============================================================================
# FINAL SUMMARY
# ============================================================================

print(f"\n" + "="*80)
print("FINAL RESULTS")
print("="*80)

summary_metrics = {
    'Accuracy': f"{accuracy_score(y_test, (y_test_pred_proba_rf > optimal_threshold).astype(int)):.1%}",
    'Precision': f"{precisions[optimal_idx]:.1%}",
    'Recall': f"{recalls[optimal_idx]:.1%}",
    'F1-Score': f"{f1_score(y_test, (y_test_pred_proba_rf > optimal_threshold).astype(int)):.1%}",
    'ROC-AUC': f"{roc_auc_score(y_test, y_test_pred_proba_rf):.3f}",
    'Customers to Target': f"{target_counts[optimal_idx]}",
    'Net Business Value': f"₹{optimal_value:,.0f}"
}

for metric, value in summary_metrics.items():
    print(f"{metric:25s}: {value}")

# ============================================================================
# SAVE PREDICTIONS
# ============================================================================

results_df = X_test.copy()
results_df['actual_churn'] = y_test.values
results_df['churn_probability'] = churn_probs
results_df['predicted_churn'] = (churn_probs > optimal_threshold).astype(int)
results_df['risk_segment'] = pd.cut(churn_probs, bins=[0, 0.35, 0.5, 0.7, 1.0],
                                      labels=['Low', 'Medium', 'High', 'Very High'])

results_df.to_csv('churn_predictions.csv', index=False)

print(f"\nFiles saved:")
print(f"  ✓ 01_feature_importance.png")
print(f"  ✓ 02_precision_recall.png")
print(f"  ✓ 03_business_value.png")
print(f"  ✓ 04_feature_contribution.png")
print(f"  ✓ churn_predictions.csv")
print(f"\nAnalysis complete.")

Data loaded: 7043 customers, 21 features
Churn rate: 26.5%

Churn by contract type:
  Month-to-month: 42.7%
  One year: 11.3%
  Two year: 2.8%

Logistic Regression - Recall: 51.3%
Training RandomForest...
RandomForest - Recall: 82.4%, ROC-AUC: 0.822

Top 3 Features:
  1. is_new_customer: 0.0664
  2. tenure_years: 0.0611
  3. TechSupport_No internet service: 0.0507

Optimal Threshold: 0.45
Customers to target: 861
Precision: 38.9%, Recall: 89.6%
Net business value: ₹161,500

Top 5 Customers at Risk:
  1. Probability: 67.6% | Actual: Churned
  2. Probability: 66.8% | Actual: Churned
  3. Probability: 66.8% | Actual: Churned
  4. Probability: 66.7% | Actual: Churned
  5. Probability: 66.6% | Actual: Churned

FINAL RESULTS
Accuracy                 : 59.9%
Precision                : 38.9%
Recall                   : 89.6%
F1-Score                 : 54.3%
ROC-AUC                  : 0.822
Customers to Target      : 861
Net Business Value       : ₹161,500

Files saved:
  ✓ 01_feature_importance